# Retrieval Validation — smarter literature retrieval

Goal: can we retrieve the papers Reactome curators cited for a gene? We build a ground-truth
set, validate the query builder, document why gene-name keyword search fails, then test a new
broad-pathway-retrieval + semantic-re-ranking pipeline.

In [ ]:
import sys
sys.path.append('../reactome_llm')
sys.path.append('..')

import json
from pathlib import Path

import ReactomeNeo4jUtils as neo4jutils

## 1. Ground-truth dataset

10 randomly-sampled, human-only validation genes and their Reactome-cited PMIDs (pathway +
reaction level). The set is **locked** — the one-time generation code below is kept for
provenance only; the live path reads back the saved `../data/ground_truth_pmids.json`.

In [ ]:
# ONE-TIME GENERATION -- do NOT re-run (resampling would break the locked set). Kept for provenance.
#
# validation_genes = neo4jutils.get_random_annotated_genes(n=10)          # human-only sampler
# ground_truth = {g: neo4jutils.query_literature_references_for_gene(g)    # human-only, pathway+reaction
#                 for g in validation_genes}
# Path('../data').mkdir(exist_ok=True)
# with open('../data/ground_truth_pmids.json', 'w') as f:
#     json.dump(ground_truth, f, indent=2)

In [ ]:
# Locked validation set -- load ground truth from JSON (no Neo4j needed).
with open('../data/ground_truth_pmids.json') as f:
    ground_truth = {gene: [int(p) for p in pmids] for gene, pmids in json.load(f).items()}

validation_genes = list(ground_truth)
for gene, pmids in ground_truth.items():
    print(f'{gene}: {len(pmids)} PMIDs')

## 2. Query-builder validation

`build_query_and_search_terms(gene)` returns `(search_terms, description)`: a PubMed-safe
OR-joined search string and a longer biological description for embedding. It is grounded with
Reactome pathway context when available, and falls back to LLM knowledge otherwise.

In [ ]:
from QueryBuilder import build_query_and_search_terms

for gene in ['TANC1', 'ELOA2']:   # TANC1: no Reactome pathways; ELOA2: has pathway context
    pathways = neo4jutils.query_pathways_for_gene(gene)
    search_terms, description = build_query_and_search_terms(gene)
    print(f'\n=== {gene} ({len(pathways)} Reactome pathways) ===')
    print('SEARCH TERMS:', search_terms)
    print('DESCRIPTION :', description)

## 3. Finding — gene-name keyword search hits a hard recall ceiling

**Setup:** narrow query (gene-name expansion) -> PubMed E-Search -> embedding re-rank.

**Result (validated across all 10 ground-truth genes):** 7/10 genes had **exactly 0** overlap
between fetched candidates (FETCH=200) and ground-truth cited PMIDs; the other 3 scored
0.008-0.053. Holds even at FETCH=500 (CD36: 6/382).

**Root cause — confirmed by sampling ground-truth abstracts directly:**
- *Dominant pattern (RASAL2, OSBPL11, ZNF540, ...):* curators cite general
  pathway/mechanism/family-level review papers as background evidence. These papers **never
  mention the specific gene by name**, so no keyword query can retrieve them.
- *Secondary pattern (CD36):* legacy / alias gene naming (GPIV, FAT, SCARB3).

Neither is fixable by better query wording — E-Search does literal keyword matching. This
motivates querying by **pathway** rather than by gene name (Section 4).

## 4. New approach — two-stage retrieval + semantic re-ranking

**Stage 1 (retrieval):** a PubMed E-Search query pulls a candidate pool.
**Stage 2 (re-rank):** embed each candidate abstract and the gene `description`, keep the top
`max_papers` by cosine similarity.

**Stage 2 is held fixed** — we compare three ways of building the *Stage-1 query*:

- **gene-name** — the bare gene symbol (baseline).
- **pathway** — the gene's Reactome pathway names (unquoted, paren-OR'd), minus mega-generic
  top-level pathways that swamp relevance ranking.
- **synonym** — the gene symbol OR'd with all UniProt protein synonyms for the gene's canonical
  protein (`build_synonym_search_query`), targeting alias-named papers (e.g. CD36 → GPIV, GP3B,
  'Fatty acid translocase'). Motivated by the CD36-style alias failures found in §3: Reactome
  reactions are defined at the protein level via UniProt, which tracks names literature uses that
  the gene symbol misses.

In [ ]:
from ReactomePubMed import ReactomePubMedRetriever
from TextEmbedder import sentence_embed, create_sentence_transformer, cosine_similarity
from QueryBuilder import (build_query_and_search_terms, build_pathway_query,
                          build_synonym_search_query, build_union_query)


# ---- Stage-1 query strategies: gene -> PubMed E-Search query string ----
# Generic-pathway filtering + the paren pathway construction live in QueryBuilder, so the
# pathway component of the union query is identical to the standalone pathway-name strategy.
def genename_query(gene):
    return gene


def pathway_query(gene):
    return build_pathway_query(gene)          # generic-filtered, capped, paren-OR'd


def synonym_query(gene):
    return build_synonym_search_query(gene)   # gene symbol + all UniProt synonyms


def union_query(gene):
    return build_union_query(gene)            # gene + synonyms + pathways, combined


STRATEGIES = {'gene-name': genename_query, 'pathway': pathway_query, 'synonym': synonym_query}


# ---- Two-stage retrieval + re-rank (Stage 2 identical across strategies) ----
def retrieve_and_rerank(gene, stage1_query_fn, fetch_papers, max_papers, model,
                        description=None, return_pool=False):
    query_text = stage1_query_fn(gene)
    if description is None:
        _, description = build_query_and_search_terms(gene)   # Stage-2 embedding target
    retriever = ReactomePubMedRetriever(top_k_results=fetch_papers)
    retriever.maxdate = '2026/03/31'                          # match frozen MongoDB cache boundary
    docs = [d for d in retriever.lazy_load(query=query_text) if d is not None]
    pool_pmids = [int(d['uid']) for d in docs]

    query_vec = sentence_embed(description, model)
    scored = []
    for d in docs:
        if not d.get('Summary'):
            continue
        scored.append((int(d['uid']), cosine_similarity(query_vec, sentence_embed(d['Summary'], model))))
    scored.sort(key=lambda x: x[1], reverse=True)
    ranked = [pmid for pmid, _ in scored[:max_papers]]
    return (ranked, pool_pmids) if return_pool else ranked

### 4a. Stage-1 only — raw candidate-pool comparison (no re-ranking)

Isolates Stage 1: for each (gene, strategy) pair, the raw counts before any re-ranking.
Four strategies are compared — the three individual ones plus their **`union`** (`build_union_query`:
gene + UniProt synonyms + generic-filtered pathway names, OR'd together).

- `GT_count` — ground-truth PMIDs for the gene
- `total_returned` — every PMID E-Search returned (**== 200 means the fetch cap was hit**, so more
  were available and a larger `FETCH_PAPERS` could help; **< 200 means E-Search was exhausted**)
- `pool_size` — real abstracts actually fetched (cache misses excluded)
- `pool_hits` — GT PMIDs found anywhere in the pool

Note: with a fixed fetch cap, the union is not guaranteed to beat the best single strategy per gene —
a very broad union (many pathways + noisy short synonyms like `FAT`) dilutes Best Match relevance and
can push good papers past the cap.

In [ ]:
import pandas as pd

# Stage-1 ONLY -- no embedding, no re-ranking.
#   total_returned = every PMID esearch returned (== fetch cap => more were available)
#   pool_size      = real abstracts actually fetched (cache misses / None excluded)
#   pool_hits      = raw count of GT PMIDs found anywhere in the pool
def stage1_counts(gene, query_fn, fetch_papers=1000):
    retriever = ReactomePubMedRetriever(top_k_results=fetch_papers)
    retriever.maxdate = '2026/03/31'
    total_returned, pool = 0, []
    for d in retriever.lazy_load(query=query_fn(gene)):
        total_returned += 1
        if d is not None:
            pool.append(int(d['uid']))
    return total_returned, pool

# 'union' added as a 4th strategy alongside the three individual ones (Stage-1 only).
STAGE1_STRATEGIES = {**STRATEGIES, 'union': union_query}

stage1_rows = []
for gene in validation_genes:
    G = set(ground_truth[gene])
    for strat, fn in STAGE1_STRATEGIES.items():
        total_returned, pool = stage1_counts(gene, fn)
        stage1_rows.append(dict(gene=gene, strategy=strat, GT_count=len(G),
                                total_returned=total_returned,
                                pool_size=len(pool), pool_hits=len(set(pool) & G)))

stage1_df = pd.DataFrame(stage1_rows).sort_values(['gene', 'strategy']).reset_index(drop=True)
stage1_df

In [ ]:
# Sanity check -- the synonym-expanded Stage-1 query for CD36 (the §3 alias example).
print(build_synonym_search_query('CD36'))

In [ ]:
import pandas as pd


def evaluate(strategy_name, query_fn, genes, ground_truth, model,
             fetch_papers=200, max_papers=50, desc_cache=None):
    """Run one Stage-1 strategy across all genes; return a per-gene precision/recall/F1 table."""
    rows = []
    for gene in genes:
        desc = desc_cache.get(gene) if desc_cache else None
        ranked, pool = retrieve_and_rerank(gene, query_fn, fetch_papers, max_papers, model,
                                           description=desc, return_pool=True)
        G = set(ground_truth[gene])
        pool_hit, top_hit = len(set(pool) & G), len(set(ranked) & G)
        prec = top_hit / max(len(ranked), 1)
        rec = top_hit / len(G)
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        rows.append(dict(strategy=strategy_name, gene=gene, GT=len(G), pool=len(pool),
                         pool_recall=pool_hit / len(G), recall=rec, precision=prec, f1=f1))
    return pd.DataFrame(rows)


FETCH_PAPERS, MAX_PAPERS = 200, 50
model = create_sentence_transformer()

# Cache the (LLM-generated) Stage-2 description once per gene; reuse across all three strategies.
desc_cache = {g: build_query_and_search_terms(g)[1] for g in validation_genes}

results = pd.concat(
    [evaluate(name, fn, validation_genes, ground_truth, model, FETCH_PAPERS, MAX_PAPERS, desc_cache)
     for name, fn in STRATEGIES.items()],
    ignore_index=True,
)

# Per-gene detail for the new synonym strategy
results[results.strategy == 'synonym'].set_index('gene')[
    ['GT', 'pool', 'pool_recall', 'recall', 'precision', 'f1']].round(3)

In [ ]:
# Head-to-head comparison, micro-averaged over all ground-truth PMIDs.
rows = []
for name in STRATEGIES:
    df = results[results.strategy == name]
    gt = df['GT'].sum()
    rows.append(dict(
        strategy=name,
        genes_with_pool_hit=int((df.pool_recall > 0).sum()),
        pool_recall=(df.pool_recall * df.GT).sum() / gt,
        final_recall=(df.recall * df.GT).sum() / gt,
        mean_precision_at50=df.precision.mean(),
        mean_f1=df.f1.mean(),
    ))
summary = pd.DataFrame(rows).set_index('strategy').round(4)
summary

### Results & interpretation — full pipeline (Stage 1 + re-rank to top 50)

Micro-averaged over all 1,333 ground-truth PMIDs (three individual strategies; see the summary
table above for exact values):

- **pathway** is the strongest Stage-1 signal (highest pool and final recall, non-zero for most genes).
- **synonym** helps only the alias cases (e.g. CD36, PLCB3) and stays near the gene-name baseline overall.
- The **re-ranker is the weak link**: it demotes the general/background papers curators cite, so final
  recall is well below pool recall for every strategy.

The Stage-1-only table (§4a) isolates retrieval from re-ranking and is the cleaner basis for comparing
which query pulls in the most ground-truth papers before any filtering.